In [3]:

import json
import os

data_dir = "data"
rows = []

for fname in sorted(os.listdir(data_dir)):
    if fname == "final_cuisine_benchmark.json":
        continue
    fpath = os.path.join(data_dir, fname)
    with open(fpath) as f:
        if fname.endswith(".jsonl"):
            count = sum(1 for line in f if line.strip())
        else:
            data = json.load(f)
            count = len(data)
    rows.append((fname, count))

# Print as a simple table
header = f"{'File':<55} {'Samples':>7}"
print(header)
print("-" * len(header))
for fname, count in rows:
    print(f"{fname:<55} {count:>7}")


File                                                    Samples
---------------------------------------------------------------
arabic_cuisine.jsonl                                     245399
arabic_cuisine_ingridients_questions.jsonl                64177
arabic_cuisine_similiarity_questions.jsonl                59000
arabic_cuisine_why_not_suitable_for_diebetics.jsonl       15630
arabic_cuisine_why_not_suitable_for_vegetarians.jsonl     31011
arabic_cuisine_why_suitable_for_diebetics.jsonl            3528
arabic_cuisine_why_suitable_for_vegetarians.jsonl          4784


In [ ]:

import json
import os

data_dir = "data"

header = f"{'File':<55} {'Keys'}"
print(header)
print("-" * 90)Instruction: ما الذي يجعل سلطة العدس بالطحينية ملائمة للنباتيين؟
Input: (empty)
Response: تُعدّ وصفة سلطة العدس... (long Arabic paragraph)

for fname in sorted(os.listdir(data_dir)):
    if fname == "final_cuisine_benchmark.json":
        continue
    fpath = os.path.join(data_dir, fname)
    with open(fpath) as f:
        if fname.endswith(".jsonl"):
            first = json.loads(next(line for line in f if line.strip()))
        else:
            data = json.load(f)
            first = data[0] if isinstance(data, list) else data
    print(f"{fname:<55} {list(first.keys())}")


File                                                    Keys
------------------------------------------------------------------------------------------
arabic_cuisine.jsonl                                    ['instruction', 'input', 'output', 'language']
arabic_cuisine_ingridients_questions.jsonl              ['instruction', 'input', 'output', 'language']
arabic_cuisine_similiarity_questions.jsonl              ['instruction', 'input', 'output', 'language']
arabic_cuisine_why_not_suitable_for_diebetics.jsonl     ['instruction', 'input', 'output', 'language']
arabic_cuisine_why_not_suitable_for_vegetarians.jsonl   ['instruction', 'input', 'output', 'language']
arabic_cuisine_why_suitable_for_diebetics.jsonl         ['instruction', 'input', 'output', 'language']
arabic_cuisine_why_suitable_for_vegetarians.jsonl       ['instruction', 'input', 'output', 'language']


In [5]:
import json
import os
import random

random.seed(42)

data_dir = "data"
TOTAL = 50_000

# --- Step 1: count each file ---
files = [
    f for f in sorted(os.listdir(data_dir))
    if f.endswith(".jsonl") and f != "final_cuisine_benchmark.json"
]

counts = {}
for fname in files:
    with open(os.path.join(data_dir, fname)) as f:
        counts[fname] = sum(1 for line in f if line.strip())

grand_total = sum(counts.values())

# --- Step 2: largest remainder method for exact 50k ---
exact = {f: TOTAL * c / grand_total for f, c in counts.items()}
floors = {f: int(v) for f, v in exact.items()}
remainder = TOTAL - sum(floors.values())
fractions = sorted(files, key=lambda f: -(exact[f] - floors[f]))
quotas = dict(floors)
for f in fractions[:remainder]:
    quotas[f] += 1

print("Quotas per file:")
for f, q in quotas.items():
    print(f"  {f:<55} {q:>6}")
print(f"  {'TOTAL':<55} {sum(quotas.values()):>6}")

# --- Step 3: sample and build pool ---
pool = []
for fname in files:
    fpath = os.path.join(data_dir, fname)
    with open(fpath) as f:
        records = [json.loads(line) for line in f if line.strip()]
    sampled = random.sample(records, quotas[fname])
    for rec in sampled:
        rec["source"] = fname
        pool.append(rec)

# --- Step 4: shuffle and assign ids ---
random.shuffle(pool)
for i, rec in enumerate(pool):
    rec["id"] = i

# reorder keys cleanly
ordered_pool = [
    {"id": rec["id"], "instruction": rec["instruction"], "input": rec["input"],
     "output": rec["output"], "language": rec["language"], "source": rec["source"]}
    for rec in pool
]

# --- Step 5: save ---
out_path = os.path.join(data_dir, "sft_pool_50k.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(ordered_pool, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(ordered_pool)} records to {out_path}")
print(f"Keys: {list(ordered_pool[0].keys())}")

Quotas per file:
  arabic_cuisine.jsonl                                     28971
  arabic_cuisine_ingridients_questions.jsonl                7576
  arabic_cuisine_similiarity_questions.jsonl                6965
  arabic_cuisine_why_not_suitable_for_diebetics.jsonl       1845
  arabic_cuisine_why_not_suitable_for_vegetarians.jsonl     3661
  arabic_cuisine_why_suitable_for_diebetics.jsonl            417
  arabic_cuisine_why_suitable_for_vegetarians.jsonl          565
  TOTAL                                                    50000

Saved 50000 records to data/sft_pool_50k.json
Keys: ['id', 'instruction', 'input', 'output', 'language', 'source']
